# SEC Form 59: executive trades

Directors and executives of Thai listed companies report every change in their holdings to the SEC
on **Form 59 (แบบ 59)**. This notebook reads that listing with `settfex`, then the report behind a
row.

Three things to keep in mind (details in `docs/settfex/services/sec/executive_trades.md`):

1. **The named executive is the reporter**, not necessarily who traded: see `executor_name`.
2. **A trade can be listed twice** when both spouses are executives (`duplicate_of`).
3. **Revoked filings stay listed** (`is_revoked`).

`economic_trades()` drops both kinds of row; count on it.

This notebook sends live requests to `market.sec.or.th`. Run it a cell at a time and keep the
windows small.

## 1. Setup

In [ ]:
from datetime import date

from settfex.services.sec import get_executive_trade_report, get_executive_trades

## 2. What SEC received on one day

With no arguments, `get_executive_trades()` returns what SEC **received today**. That is not the
page the site shows by default, which lists what was *recorded* today. Here we ask for one past day.

In [ ]:
day = await get_executive_trades(date(2026, 10, 2))
print(repr(day))
print("stated by the site:", day.reported_count, "| parsed:", len(day))
print("revoked:", day.revoked_count, "| duplicates:", day.duplicate_count)
print("unknown labels:", day.unknown_labels)

## 3. The summary table

`to_table()` renders the Thai table: sells first, then buys; `**` marks a trade by someone other
than the reporter. Revoked rows are skipped.

In [ ]:
print(day.economic_trades().to_table())

## 4. One company, by trade date

`symbol` is resolved strictly to the SEC's company id. `date_type="transaction"` filters on the
trade date instead of the received date.

In [ ]:
spali = await get_executive_trades(
    symbol="SPALI", date_type="transaction", start="2026-09-01", end="2026-09-30"
)
for t in spali[:6]:
    print(
        t.transaction_date,
        t.reporter_name,
        "->",
        t.executor_name or "(self)",
        t.method,
        f"{t.quantity:,}",
        t.price,
        "| copy of",
        t.duplicate_of,
    )

## 5. The report behind a row: holdings before and after

Each row's `batch_no` names its report. One call returns every transaction in it, with holdings,
the unrounded average price, the trading channel and broker, and the counterparty.

In [ ]:
report = await get_executive_trade_report("592001352610")
print(
    report.company_name, "|", report.reporter_name, "|", report.position, "|", report.submitted_at
)
for tx in report.transactions:
    print(
        tx.transaction_date,
        tx.method,
        f"{tx.holding_before:,} -> {tx.holding_after:,}",
        tx.avg_price,
        tx.market_source,
        "| consistent:",
        tx.holding_consistent,
    )

## 6. Enriching a listing

`with_details=True` fetches each distinct report once (at most 100 per call) and attaches the
matching transaction to each row. Duplicates are then decided by holdings.

In [ ]:
enriched = await get_executive_trades(
    symbol="SPALI",
    date_type="transaction",
    start="2026-09-29",
    end="2026-09-30",
    with_details=True,
)
print(enriched.to_table(with_details=True))
print("detail failures:", enriched.detail_failures, "| unmatched:", enriched.detail_unmatched)

## 7. As a DataFrame (needs `pip install settfex[dataframe]`)

In [ ]:
frame = enriched.to_dataframe()
frame[
    [
        "symbol",
        "reporter_name",
        "executor_name",
        "side",
        "quantity",
        "price",
        "holding_before",
        "holding_after",
        "duplicate_of",
    ]
]